# 2025-04-28-利用奇异值重构矩阵-美团

## 题目内容

在一家致力于图像处理的科技公司，你被分配到一个新项目，目标是开发一种图像压缩算法，以减少存储空间并加速传输。团队决定使用奇异值分解（$SVD$）对图像进行降维处理，以达到压缩的目的。

现在，你需要编写一个程序，对给定的灰度图像矩阵进行奇异值分解，并重构出近似的低秩矩阵。

请你帮助团队实现一个使用$NumPy$库的程序，对给定的矩阵进行奇异值分解，共利用前$(k)$个奇异值重构矩阵。具体要求如下：
1. 读取输入矩阵，为一个二维列表，表示灰度图像的像素值矩阵。
2. 读取整数$(k)$，表示使用前$(k)$个奇异值进行矩阵重构。
3. 对矩阵进行奇异值分解($SVD$)，获取左奇异矩阵$(U)$、奇异值对角矩阵$(\Sigma)$和右奇异矩阵$(V^T)$。
4. 利用前$(k)$个奇异值和对应的奇异向量重构矩阵。
5. 输出重构后的矩阵，每个元素保留两位小数（使用$round(x, 2)$）。

## 输入描述

- 第一行包含两个整数$(m)$和$(n)$表示矩阵的行数和列数。
- 接下来的$(m)$行，每行包含$(n)$个整数，表示矩阵的元素，元素之间用空格分隔。
- 最后一行包含一个整数$(k)$，表示用前$(k)$个奇异值进行矩阵重构。

## 输出描述

输出$(m)$行，每行包含$(n)$个浮点数，表示重构后的矩阵元素，元素之间用空格分隔，结果均保留两位小数，使用$round(x, 2)$。


## 补充说明

- 奇异值分解（$SVD$），对于一个矩阵 $(A)$ 其奇异值分解为：
  $$
  A = U \Sigma V^T
  $$
  其中：4 5
  - $(U)$ 是 $(m \times m)$ 的正交矩阵，其列为 $(AA^T)$ 的特征向量。
  - $(\Sigma)$ 是 $(m \times n)$ 的对角矩阵，对角线上为非负的奇异值，按降序排列。
  - $(V^T)$ 是 $(n \times n)$ 的正交矩阵的转置，其列为 $(A^TA)$ 的特征向量。

- 矩阵重构

  使用前 $(k)$ 个奇异值和对应的奇异向量，可以近似地重构原矩阵：
  $$
  A_k = U_k \Sigma_k V_k^T
  $$
  其中：
  - $(U_k)$ 为 $(U)$ 的前 $(k)$ 列。
  - $(\Sigma_k)$ 为 $(\Sigma)$ 中前 $(k)$ 个奇异值构成的对角矩阵。
  - $(V_k^T)$ 为 $(V^T)$ 的前 $(k)$ 行。


## 样例
```
输入:
4 5
52 55 61 66 70
63 59 55 90 109
85 104 117 123 119
105 122 145 160 159
2

输出:
47.3 54.1 61.41 69.2 70.35
63,02 58.35 55.4 90.03 109.11
84.08 101.15 118.72 123.8 119.52
107.77 125.0 143.24 157.95 158.38

```

In [217]:
import numpy as np

In [219]:
n,m = map(int, input().split())
mat = []
for _ in range(n):
    mat.append(list(map(int, input().split())))
k = int(input())

In [242]:
A = np.array(mat)
A

array([[ 52,  55,  61,  66,  70],
       [ 63,  59,  55,  90, 109],
       [ 85, 104, 117, 123, 119],
       [105, 122, 145, 160, 159]])

In [254]:
eig_values, U = np.linalg.eig(A@(A.T))
U

array([[ 0.30050372, -0.02694092,  0.69020128,  0.6577187 ],
       [ 0.37778194,  0.90391529,  0.05467   , -0.19294855],
       [ 0.54241183, -0.38739585,  0.36703081, -0.64884685],
       [ 0.68758106, -0.17926459, -0.62122602,  0.33041599]])

In [255]:
eig_values = sorted(np.sqrt(eig_values),reverse=True)
eig_values 

[454.71798467974327, 34.01883573921087, 7.740636522193636, 3.369832336013565]

In [256]:
_, V_t = np.linalg.eig((A.T)@A)
V_t = V_t.T
V_t

array([[ 0.34686878,  0.39389812,  0.44482554,  0.50704668,  0.51919189],
       [-0.11153546,  0.30307318,  0.68334892, -0.09530755, -0.64780955],
       [-0.68515863, -0.46097385,  0.26174063,  0.48804619,  0.1065998 ],
       [ 0.41943452, -0.20543104, -0.25978011,  0.65168322, -0.53823419],
       [ 0.47103526, -0.70589809,  0.44628536, -0.26634569,  0.09860531]])

In [257]:
U_k = U[:, :k]
sigma = np.diag(eig_values[:k])
V_tk = V_t[:k, :]
A_k = np.round(U_k@sigma@V_tk, 2)
A_k

array([[ 47.5 ,  53.55,  60.16,  69.37,  71.54],
       [ 56.16,  76.99,  97.43,  84.17,  69.27],
       [ 87.02,  93.16, 100.71, 126.32, 136.59],
       [109.13, 121.31, 134.91, 159.11, 166.28]])

## 基础概念

### 求奇异值

奇异值可以通过以下步骤计算：

1. 计算矩阵 $A$ 的转置 $A^T$。
2. 计算 $A^TA$ 和 $AA^T$。
3. 求解 $A^TA$ 和 $AA^T$ 的特征值。
4. 奇异值是 $A^TA$ 和 $AA^T$ 的特征值的平方根。

### numpy求矩阵特征值

In [258]:
import numpy as np

# 定义一个矩阵
A = np.array([[1, 2], [3, 4]])

# 使用 numpy.linalg.eig 计算特征值和特征向量
eigenvalues, eigenvectors = np.linalg.eig(A)

# 打印结果
print("矩阵A：")
print(A)
print("\n特征值：")
print(eigenvalues)
print("\n特征向量：")
print(eigenvectors)

矩阵A：
[[1 2]
 [3 4]]

特征值：
[-0.37228132  5.37228132]

特征向量：
[[-0.82456484 -0.41597356]
 [ 0.56576746 -0.90937671]]


In [259]:
eigenvalues = np.linalg.eigvals(A)
print("特征值：", eigenvalues)

特征值： [-0.37228132  5.37228132]


### numpy求奇异值分解

直接用这个算出的答案不太一样

In [260]:
import numpy as np

# 创建一个示例矩阵
A = np.array([[1, 2], [3, 4], [5, 6]])

# 进行奇异值分解
U, S, Vh = np.linalg.svd(A, full_matrices=True, compute_uv=True)

print("U 矩阵 (左奇异向量):")
print(U)
print("\n奇异值 (S):")
print(S)
print("\nVh 矩阵 (右奇异向量的转置):")
print(Vh)

U 矩阵 (左奇异向量):
[[-0.2298477   0.88346102  0.40824829]
 [-0.52474482  0.24078249 -0.81649658]
 [-0.81964194 -0.40189603  0.40824829]]

奇异值 (S):
[9.52551809 0.51430058]

Vh 矩阵 (右奇异向量的转置):
[[-0.61962948 -0.78489445]
 [-0.78489445  0.61962948]]
